# Ven 16/10 · 15:45 et 17:30 — TP 3.1 : ligne de commande, logging et codes de sortie

Ven 16/10, en deux parties :

| Horaire | Contenu |
|---|---|
| 16:45–17:15 | Partie 1 : ligne de commande |
| 17:55–18:20 | Partie 2 : logging et codes de sortie |
| 18:20–18:30 | Correction des deux parties, TP fini ou non |

À 18:30, IDE fermé : QCM de révision sur Google Forms, lien dans le chat Teams. La référence du TP 3.1 est publiée à 19:00.

La partie 2 part de votre partie 1 : finissez-la d'abord. En partie 2, priorité aux étapes 1 et 2 : l'étape 3 peut se finir chez vous, à partir de la référence du TP 3.1.

## Objectif

`fleetcheck` devient une commande utilisable en CI :

- ses options `--inventory`, `--env` et `--format` remplacent la constante `INVENTORY` du TP 2.2 ;
- le journal (lignes ignorées, erreurs) passe par `logging` sur stderr, et stdout ne contient plus que le résultat ;
- `-v` règle la quantité de journal ;
- le code de sortie dit si le contrôle est fiable : `0` succès, `1` panne détectée, `2` erreur fatale ou ligne d'inventaire invalide.

## Mode d'emploi du notebook

Celui de `tp/tp1.1.ipynb` vaut toujours. Nouveau depuis le TP 2.2 :

- Dès la partie 1, le script a un bloc `if __name__ == "__main__":` qui appelle `main()` ; `%run -n fleetcheck.py` charge les fonctions sans l'exécuter. N'appelez pas `main()` dans le notebook : argparse y lirait les arguments du noyau Jupyter et s'arrêterait.
- `run(...)` accepte les arguments du script : `run("--env", "staging")` équivaut à `python3 fleetcheck.py --env staging`. Il lance le script comme dans un terminal et renvoie `r.stdout`, `r.stderr` et `r.returncode`. Journal et codes de sortie se vérifient par `run(...)` : `logging.basicConfig` ne s'applique qu'une fois par noyau, chaque `run(...)` repart d'un processus neuf.
- Les vérifications qui retirent ou créent un fichier remettent tout en place : ne les modifiez pas.
- Rendu : *Restart Kernel and Run All Cells*, puis commit et push de `fleetcheck.py`, `config.json`, `servers.csv`, `servers-broken.csv` et de ce notebook.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args, quiet=False):
    """Lance fleetcheck.py comme dans un terminal (vu au TP 4.3 : subprocess). quiet=True n'affiche pas stdout."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    if not quiet:
        print(r.stdout, end="", flush=True)
    print(r.stderr, end="", file=sys.stderr, flush=True)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

## Point de départ

Votre `fleetcheck.py` du TP 2.2, avec `config.json`, `servers.csv` et `servers-broken.csv` dans le même dossier.

En retard ou en panne : copiez dans votre dépôt les fichiers de `reference/` au tag `tp2.2` du dépôt public (`git switch --detach tp2.2` dans votre clone de consultation, voir le `README.md` à la racine).

Vérifiez avant de commencer, avec `INVENTORY = "servers.csv"` :

In [ ]:
r = run()
check(r.stdout, """9 serveurs valides, 0 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
""")

## Partie 1 — ligne de commande

`main()` lit ses options dans `sys.argv`. Dans le notebook, `sys.argv` contient les arguments du noyau Jupyter (`-f …kernel.json`) : appeler `main()` ou exécuter `%run fleetcheck.py` sans `-n` fait échouer argparse. Les fonctions se vérifient après `%run -n`, la ligne de commande avec `run(...)`.

### 1. `main()` et le parser — fin 16:58

Écrivez `main()` avec un parser `argparse` à 3 options :

| Option | Valeur |
|---|---|
| `--inventory` | fichier CSV, défaut `servers.csv` |
| `--env` | facultatif, un environnement |
| `--format` | `text` (défaut) ou `json`, rien d'autre |

Chaque option a un `help`. La constante `INVENTORY` disparaît. Le résumé texte passe dans une fonction `print_summary(servers, invalid)`, que `main()` appelle. Le script appelle `main()` sous `if __name__ == "__main__":`.

Vérification : l'aide liste `--inventory`, `--env` et `--format`, et sort en `0` ; `print_summary` affiche ce qu'affiche `run()`.

In [ ]:
r = run("--help")
check([option in r.stdout for option in ("--inventory", "--env", "--format")], [True, True, True])
check(r.returncode, 0)

%run -n fleetcheck.py
import contextlib
import io
servers, invalid = load("servers.csv")
sortie = io.StringIO()
with contextlib.redirect_stdout(sortie):
    print_summary(servers, invalid)
check(sortie.getvalue(), run(quiet=True).stdout)

### 2. `--env` — fin 17:05

`--env` filtre les serveurs valides avant le résumé et le rapport. Sans `--env`, rien ne change.

Vérification : la dernière ligne s'arrête après `prod/web : ` (espace final compris) : aucun serveur `prod` ne reste après le filtre.

In [ ]:
r = run("--env", "staging")
check(r.stdout, """2 serveurs valides, 0 invalides
par env : staging 2
par rôle : web 1, api 1
prod/web : 
""")

### 3. `--format json` — fin 17:15

`--format json` affiche le contenu du rapport sur stdout, avec `json.dumps(report, indent=2, ensure_ascii=False)`. `--format text` garde le résumé. `report.json` est écrit dans les deux cas.

Vérification : la sortie standard se relit avec `json.loads`, et c'est le contenu de `report.json`.

In [ ]:
import json

r = run("--format", "json")
try:
    report = json.loads(r.stdout)
except json.JSONDecodeError as e:
    report = {"summary": None}
    print(f"KO : stdout n'est pas un JSON seul ({e})")
check(report["summary"], {"total": 9, "invalid": 0})
with open("report.json", encoding="utf-8") as f:
    check(json.load(f), report)
check(r.returncode, 0)

**Prédire** — `run("--format", "xml")` : votre code s'exécute-t-il ? Quel code de sortie, et sur quel flux arrive le message ?

In [ ]:
# Ma réponse :

Un format inconnu : argparse affiche l'usage puis une erreur sur stderr, et sort en `2` avant que votre code ne tourne. Dernière ligne attendue, avec Python 3.14 :

```text
fleetcheck.py: error: argument --format: invalid choice: 'xml' (choose from text, json)
```

In [ ]:
r = run("--format", "xml")
check(r.stderr.splitlines()[-1:], ["fleetcheck.py: error: argument --format: invalid choice: 'xml' (choose from text, json)"])
check(r.returncode, 2)

Enfin, `--inventory servers-broken.csv` redonne la sortie attendue du TP 2.2 : le résumé, puis les lignes `invalide ligne 4`, `8` et `12`. C'est le critère de fin de la partie 1. La cellule vérifie le résumé, et que les 3 lignes invalides sont citées : en partie 2, elles passent sur stderr.

In [ ]:
r = run("--inventory", "servers-broken.csv")
check(r.stdout.splitlines()[:4], ["9 serveurs valides, 3 invalides",
                                  "par env : prod 7, staging 2",
                                  "par rôle : web 3, api 3, worker 1, cache 1, db 1",
                                  "prod/web : web-1, web-2"])
check([f"ligne {n} " in r.stdout + r.stderr for n in (4, 8, 12)], [True, True, True])

### À observer

**Prédire** — `--env prd` est une faute de frappe pour `prod`. Que va afficher `run("--env", "prd")`, avec quel code de sortie ? Un job de CI qui lance cette commande finit-il vert ou rouge, et est-ce le bon résultat ?

In [ ]:
# Ma réponse :

In [ ]:
r = run("--env", "prd")

`--env prd` affiche `0 serveurs valides, 0 invalides` et sort en `0` : une CI serait verte sans avoir rien contrôlé. La partie 2 corrige ce défaut.

## Partie 2 — logging et codes de sortie

| Fin | Étape |
|---|---|
| 18:06 | 1. Remplacer les `print` d'erreur par `logging` sur stderr : `WARNING` pour une ligne invalide, `ERROR` pour une erreur fatale. `-v` passe en `INFO` |
| 18:12 | 2. Fonction `exit_code(statuses, invalid_count)` |
| 18:20 | 3. Erreurs fatales en `ERROR` et code `2` ; code de sortie final calculé par `exit_code` |

À 18:12, si plus d'un tiers de la salle n'a pas `exit_code` (✅/❌ dans le chat), l'enseignant l'écrit en partage d'écran (6 lignes, 2 min).

### 1. Logging sur stderr — fin 18:06

```mermaid
flowchart LR
  fc["fleetcheck.py"] -- "stdout : le résultat, résumé ou JSON" --> out["r.stdout"]
  fc -- "stderr : le journal, de DEBUG à ERROR" --> err["r.stderr"]
```

- Un logger nommé `fleetcheck`, configuré une seule fois au début de `main()` : format `%(levelname)s %(message)s`, sortie stderr.
- Ligne invalide : `WARNING ligne N ignorée : <erreur>`. Les lignes `invalide ligne N : ...` quittent le résumé, qui garde ses 4 lignes.
- Erreur fatale des TP 2.1 et 2.2 (`print` suivi de `sys.exit`) : `ERROR`. Le code de sortie change à l'étape 3.
- Option `-v` / `--verbose` (`action="count"`) : niveau `WARNING` sans option, `INFO` avec `-v`. Au moins un message `INFO` par exécution, au texte libre (par exemple le nombre de lignes lues).
- Arguments du message passés à part, pas en f-string : `log.warning("ligne %d ignorée : %s", line, e)`.

**Justifier** — Si les `WARNING` restaient sur stdout, que recevrait `jq` dans `python3 fleetcheck.py --inventory servers-broken.csv --format json | jq .` ?

In [ ]:
# Ma réponse :

- `servers-broken.csv` : stdout garde le résumé, stderr porte les 3 `WARNING` au texte exact.

In [ ]:
r = run("--inventory", "servers-broken.csv")
check(r.stdout, """9 serveurs valides, 3 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
""")
check(r.stderr, """WARNING ligne 4 ignorée : port non entier : 'abc'
WARNING ligne 8 ignorée : IP invalide : '300.0.0.1'
WARNING ligne 12 ignorée : 3 champs au lieu de 5
""")

- Sans option, sur `servers.csv` : rien sur stderr.

In [ ]:
r = run()
check(r.stderr, "")

- Avec `-v` : chaque ligne de stderr commence par un niveau, au moins une par `INFO`, aucune par `DEBUG` ; stdout ne contient que le résumé.

In [ ]:
NIVEAUX = ("DEBUG ", "INFO ", "WARNING ", "ERROR ", "CRITICAL ")
RESUME = """9 serveurs valides, 0 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
"""

r = run("-v")
journal = r.stderr.splitlines()
check(all(ligne.startswith(NIVEAUX) for ligne in journal), True)
check(any(ligne.startswith("INFO ") for ligne in journal), True)
check(any(ligne.startswith("DEBUG ") for ligne in journal), False)
check(r.stdout, RESUME)

### 2. `exit_code` — fin 18:12

`exit_code(statuses, invalid_count)` reçoit une liste de statuts de services (`"ok"`, `"degraded"`, `"down"`) et le nombre de lignes invalides. Elle renvoie :

```mermaid
flowchart LR
  inv(["invalid_count non nul ?"]) -- oui --> c2["2"]
  inv -- non --> down(["au moins un statut down ?"])
  down -- oui --> c1["1"]
  down -- non --> c0["0"]
```

La fonction est pure : ni `print`, ni `sys.exit`, ni lecture de fichier. Aucun service n'est encore contrôlé (le contrôle réseau arrive au TP 4.3) : `main()` l'appelle avec une liste vide.

In [ ]:
%run -n fleetcheck.py


**Justifier** — Une ligne d'inventaire invalide l'emporte sur un service `down` : `exit_code(["down"], 1)` vaut `2`. Pourquoi cet ordre ?

In [ ]:
# Ma réponse :

Vérification : `1 2 0`, puis `2` quand une ligne invalide et un `down` coexistent.

In [ ]:
%run -n fleetcheck.py
check((exit_code(["ok", "down"], 0), exit_code(["ok"], 1), exit_code([], 0)), (1, 2, 0))
check(exit_code(["down"], 1), 2)

### 3. Codes de sortie — fin 18:20

Erreurs fatales : log `ERROR`, puis `sys.exit(2)`, sans écrire `report.json`.

| Cas | Message de la référence |
|---|---|
| Inventaire absent | `ERROR inventaire illisible : [Errno 2] No such file or directory: 'nope.csv'` |
| Config absente ou JSON invalide | `ERROR config illisible (config.json) : ...` |
| Aucune ligne valide dans l'inventaire | `ERROR aucune ligne valide dans <fichier>` |
| `--env` ne garde aucun serveur | `ERROR aucun serveur pour --env <valeur>` |

Un fichier présent mais illisible (un dossier, des droits manquants, un fichier qui n'est pas en UTF-8) est aussi une erreur fatale : élargissez `FileNotFoundError` à `OSError` et `UnicodeDecodeError` (qui n'hérite pas d'`OSError`). Sinon le script plante avec une trace et sort en `1`, le code d'une panne.

Fin de `main()` : `sys.exit(exit_code([], len(invalid)))`. Une ligne invalide donne donc `2`, mais le rapport est écrit et le résumé affiché : les serveurs valides restent traités.

**Prédire** — `sys.exit` lève une exception, `SystemExit`. Dans un terminal bash, qu'affiche `python3 sortie.py && echo OK` ? Et `python3 sortie.py | tee sortie.log && echo OK` ?

```text
# sortie.py
import sys

try:
    sys.exit(2)
finally:
    print("fin")
```

In [ ]:
# Ma réponse :

Vérification de l'affichage et du code de sortie, dans un processus à part comme en terminal (le pipe se vérifie à la correction, dans bash) :

In [ ]:
import subprocess
import sys
from pathlib import Path

Path("sortie.py").write_text('import sys\n\ntry:\n    sys.exit(2)\nfinally:\n    print("fin")\n', encoding="utf-8")
try:
    r = subprocess.run([sys.executable, "sortie.py"], capture_output=True, encoding="utf-8")
finally:
    Path("sortie.py").unlink()
check((r.stdout, r.returncode), ("fin\n", 2))

Vérification des erreurs fatales : stdout vide, une ligne `ERROR` en dernier sur stderr, code `2`, pas de `report.json`. La cellule supprime `report.json` avant chaque cas, retire `config.json` le temps d'un run, crée un inventaire sans ligne valide et un inventaire vide (0 octet) ; elle remet tout en place.

In [ ]:
import shutil
from pathlib import Path

def fatal(*args):
    Path("report.json").unlink(missing_ok=True)
    r = run(*args)
    return r.stdout, r.stderr.splitlines()[-1:], r.returncode, Path("report.json").exists()

check(fatal("--inventory", "nope.csv"),
      ("", ["ERROR inventaire illisible : [Errno 2] No such file or directory: 'nope.csv'"], 2, False))

stdout, stderr, code, rapport = fatal("--inventory", ".")
check((stdout, stderr[0].startswith("ERROR inventaire illisible : "), code, rapport), ("", True, 2, False))

Path("latin1.csv").write_bytes("name,env,ip,port,role\nwéb-1,prod,127.0.0.1,8001,web\n".encode("latin-1"))
try:
    stdout, stderr, code, rapport = fatal("--inventory", "latin1.csv")
finally:
    Path("latin1.csv").unlink()
check((stdout, "".join(stderr).startswith("ERROR inventaire illisible : "), code, rapport), ("", True, 2, False))

shutil.move("config.json", "config.bak")
try:
    resultat = fatal()
finally:
    shutil.move("config.bak", "config.json")
check(resultat, ("", ["ERROR config illisible (config.json) : [Errno 2] No such file or directory: 'config.json'"], 2, False))

Path("aucune-valide.csv").write_text("name,env,ip,port,role\napi-3,prod,127.0.0.1,abc,api\n", encoding="utf-8")
try:
    resultat = fatal("--inventory", "aucune-valide.csv")
finally:
    Path("aucune-valide.csv").unlink()
check(resultat, ("", ["ERROR aucune ligne valide dans aucune-valide.csv"], 2, False))

Path("vide.csv").write_text("", encoding="utf-8")
try:
    resultat = fatal("--inventory", "vide.csv")
finally:
    Path("vide.csv").unlink()
check(resultat, ("", ["ERROR en-tête attendu dans vide.csv : name,env,ip,port,role"], 2, False))

## Critère final

Quatre lancements, l'équivalent de ce qu'une CI exécuterait :

| Lancement | stdout | stderr | Code |
|---|---|---|---|
| `run()` | le résumé | rien | `0` |
| `run("--inventory", "servers-broken.csv")` | le résumé | les 3 `WARNING` | `2` |
| `run("--env", "prd")` | rien | `ERROR aucun serveur pour --env prd` | `2` |
| `run("--format", "json")` | un JSON valide | rien | `0` |

In [ ]:
r = run()
check((r.stdout, r.stderr, r.returncode), (RESUME, "", 0))

In [ ]:
r = run("--inventory", "servers-broken.csv")
check((r.stderr.count("WARNING ligne"), r.returncode), (3, 2))

In [ ]:
r = run("--env", "prd")
check((r.stdout, r.stderr, r.returncode), ("", "ERROR aucun serveur pour --env prd\n", 2))

In [ ]:
import json

r = run("--format", "json")
check((json.loads(r.stdout)["summary"], r.stderr, r.returncode), ({"total": 9, "invalid": 0}, "", 0))

En fin de partie 1, `--env prd` affichait `0 serveurs valides, 0 invalides` et sortait en `0` : une faute de frappe dans un job de CI laissait le pipeline vert sans rien contrôler.

## Pour aller plus loin

- `-vv` passe en `DEBUG` : un message `DEBUG` par ligne valide, au texte libre. Exemple de la référence :

```text
DEBUG ligne 2 valide : web-1
DEBUG ligne 3 valide : web-2
...
DEBUG ligne 10 valide : db-1
INFO 9 serveur(s) valide(s), 0 ligne(s) invalide(s) dans servers.csv
INFO rapport écrit : report.json
```

Vérification : chaque ligne de stderr commence par un niveau, au moins une par `DEBUG`, stdout inchangé. Sans ligne `DEBUG`, la cellule l'indique sans `KO` : l'extension n'est pas exigée au rendu.

In [ ]:
r = run("-vv")
journal = r.stderr.splitlines()
if any(ligne.startswith("DEBUG ") for ligne in journal):
    check(all(ligne.startswith(NIVEAUX) for ligne in journal), True)
    check(r.stdout, RESUME)
else:
    print("Aucune ligne DEBUG : extension -vv non traitée")